# Experiment 1-3 — Baseline, Logit Lens, Causal Tracing

**Group 25 · Opening the Black Box**

This notebook uses the project package (`models/`, `data/`) to run the *interpretability*
experiments that the editing step (notebook `04_editing_and_generation.ipynb`) builds on:

1. **Baseline (Exp 1)** — logit difference between the correct and incorrect answer on each suite.
2. **Logit lens (Exp 2)** — watch a factual prediction take shape across layers.
3. **Causal tracing (Exp 3)** — localize the MLP that stores a fact (this feeds the edit).

Run cells top to bottom. First run installs `torch` / `transformer_lens` and downloads `pythia-160m`.

In [ ]:
# Setup + make the repo importable
import importlib, subprocess, sys, os
for imp, pip in [("torch","torch"), ("transformers","transformers"), ("transformer_lens","transformer_lens")]:
    try: importlib.import_module(imp)
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install","-q",pip])

sys.path.insert(0, os.path.abspath(".."))    # repo root, so `import models` works from experiments/
from models import (load_config, load_model, suite_logit_diffs, logit_lens,
                    causal_trace, best_edit_layer)
from data import load_behavior_suite, SUITES
import pandas as pd, numpy as np, matplotlib.pyplot as plt

cfg = load_config()
model = load_model(cfg)

## 1. Baseline logit differences (Experiment 1)

In [ ]:
frames = []
for name in SUITES:
    df = pd.DataFrame(suite_logit_diffs(model, load_behavior_suite(name)))
    df["suite"] = name
    frames.append(df)
    print(f"{name:16s} mean logit diff = {df['logit_diff'].mean():+.3f}  "
          f"accuracy = {(df['logit_diff']>0).mean():.0%}")
baseline = pd.concat(frames, ignore_index=True)
baseline.groupby("suite")["logit_diff"].describe()[["mean","min","max"]]

## 2. Logit lens (Experiment 2)

Per-layer log-prob of the correct answer when each layer's residual stream is decoded directly.

In [ ]:
prompt, ans = "The capital of France is", " Paris"
curve = logit_lens(model, prompt, ans)
plt.figure(figsize=(7,4))
plt.plot(range(len(curve)), curve, marker="o")
plt.xlabel("layer"); plt.ylabel(f"log-prob of '{ans.strip()}'")
plt.title(f"Logit lens — {prompt!r}"); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 3. Causal tracing (Experiment 3)

Corrupt the subject, patch clean states back layer-by-layer, and read off where the fact lives. The peak at the subject's last token is the layer we edit in notebook 04.

In [ ]:
recovery, subj_last, str_toks = causal_trace(model, "The Eiffel Tower is located in the city of",
                                             "The Eiffel Tower", "Paris", cfg)
fig, ax = plt.subplots(figsize=(9,4))
im = ax.imshow(recovery, aspect="auto", cmap="magma", vmin=0, vmax=1)
ax.set_xticks(range(len(str_toks)))
ax.set_xticklabels([t.strip() or "·" for t in str_toks], rotation=60, ha="right", fontsize=8)
ax.set_yticks(range(model.cfg.n_layers)); ax.set_ylabel("layer")
ax.set_title("Causal-trace recovery of the true-object probability")
plt.colorbar(im, label="fraction restored"); plt.tight_layout(); plt.show()
print("Best edit layer:", best_edit_layer(recovery, subj_last))

## Next
- `scripts/run_baseline.py` and `scripts/run_ablation.py` run Experiments 1 and 3a headless.
- `04_editing_and_generation.ipynb` (or `scripts/run_edit.py`) performs the ROME-style edit,
  generates under it, and scores efficacy / generalization / specificity / fluency.